<a href="https://colab.research.google.com/github/mandarveer/EV-BMS/blob/main/notebooks/BMS_with_live_chart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# SMART EV BMS + VEHICLE ENERGY MANAGEMENT SYSTEM
# GOOGLE COLAB INTERACTIVE DASHBOARD
# ============================================================
#
# Features:
#
# Battery:
#   SOC
#   SOH
#   Battery capacity
#   Battery temperature
#
# Vehicle:
#   Speed
#   Payload
#   Drive mode
#
# Environment:
#   Outdoor temperature
#   AC temperature
#
# Tires:
#   FL
#   FR
#   RL
#   RR
#
# Outputs:
#   Battery power
#   Energy consumption
#   Estimated range
#   Remaining range
#   Battery current
#   Battery stress
#   Warnings
#   Alerts
#   Individual tire status
#   Drive-mode effect
#
# ============================================================


# ============================================================
# 1. INSTALL LIBRARIES
# ============================================================

!pip install -q gradio numpy pandas


# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import numpy as np
import pandas as pd
import gradio as gr
import math


# ============================================================
# 3. VEHICLE PARAMETERS
# ============================================================

VEHICLE_MASS = 1600          # kg
CD = 0.29                    # Drag coefficient
FRONTAL_AREA = 2.25          # m²
AIR_DENSITY = 1.225          # kg/m³
GRAVITY = 9.81               # m/s²

DRIVETRAIN_EFFICIENCY = 0.90

REFERENCE_TIRE_PRESSURE = 34.0

NOMINAL_PACK_VOLTAGE = 400.0


# ============================================================
# 4. DRIVE MODE PARAMETERS
# ============================================================

DRIVE_MODES = {

    "Eco": {
        "power_factor": 0.85,
        "hvac_factor": 0.85,
        "aux_factor": 0.85,
        "regen_factor": 1.10,
        "stress_factor": 0.85
    },

    "Normal": {
        "power_factor": 1.00,
        "hvac_factor": 1.00,
        "aux_factor": 1.00,
        "regen_factor": 1.00,
        "stress_factor": 1.00
    },

    "Sport": {
        "power_factor": 1.15,
        "hvac_factor": 1.05,
        "aux_factor": 1.10,
        "regen_factor": 0.90,
        "stress_factor": 1.20
    }
}


# ============================================================
# 5. TIRE PRESSURE MODEL
# ============================================================

def tire_pressure_factor(pressure):

    pressure = max(0, min(40, pressure))

    if pressure < 15:
        return 1.60

    elif pressure < 20:
        return 1.40

    elif pressure < 25:
        return 1.22

    elif pressure < 30:
        return 1.10

    elif pressure < 32:
        return 1.04

    elif pressure <= 36:
        return 1.00

    elif pressure <= 38:
        return 1.03

    else:
        return 1.08


# ============================================================
# 6. INDIVIDUAL TIRE STATUS
# ============================================================

def individual_tire_status(pressure):

    if pressure < 15:
        return "🔴 CRITICAL"

    elif pressure < 25:
        return "🟠 LOW"

    elif pressure < 30:
        return "🟡 BELOW OPTIMAL"

    elif pressure <= 36:
        return "🟢 NORMAL"

    elif pressure <= 38:
        return "🟡 HIGH"

    else:
        return "🔴 VERY HIGH"


# ============================================================
# 7. HVAC MODEL
# ============================================================

def calculate_hvac_power(
        outdoor_temp,
        ac_temp,
        drive_mode):

    mode = DRIVE_MODES[drive_mode]

    temperature_difference = abs(
        outdoor_temp - ac_temp
    )

    # Base HVAC consumption
    base_power = 0.60

    # Temperature conditioning load
    thermal_load = (
        0.08 * temperature_difference
    )

    # Extreme weather penalty
    extreme_load = 0

    if outdoor_temp >= 40:
        extreme_load = 1.0

    elif outdoor_temp >= 35:
        extreme_load = 0.5

    elif outdoor_temp <= 5:
        extreme_load = 0.8

    hvac_power = (
        base_power
        + thermal_load
        + extreme_load
    )

    hvac_power *= mode["hvac_factor"]

    return hvac_power


# ============================================================
# 8. BATTERY THERMAL LOAD
# ============================================================

def battery_thermal_load(
        battery_temp,
        outdoor_temp):

    load = 0

    # Cold battery
    if battery_temp < 0:
        load = 2.5

    elif battery_temp < 10:
        load = 1.5

    elif battery_temp < 20:
        load = 0.5

    # Normal region
    elif battery_temp <= 35:
        load = 0

    # Warm battery
    elif battery_temp <= 40:
        load = 0.5

    elif battery_temp <= 45:
        load = 1.5

    # Critical temperature
    else:
        load = 3.0

    # Extreme ambient temperature
    if outdoor_temp > 40:
        load += 0.5

    return load


# ============================================================
# 9. BATTERY TEMPERATURE STATUS
# ============================================================

def battery_temperature_status(temp):

    if temp < 0:

        return (
            "🔵 COLD",
            "Battery temperature below 0°C."
        )

    elif temp < 10:

        return (
            "🔵 LOW",
            "Battery is cold. Power and charging "
            "performance may be reduced."
        )

    elif temp < 20:

        return (
            "🟡 COOL",
            "Battery temperature is below the "
            "preferred operating range."
        )

    elif temp <= 35:

        return (
            "🟢 NORMAL",
            "Battery temperature is within "
            "the preferred operating range."
        )

    elif temp <= 40:

        return (
            "🟡 WARM",
            "Battery temperature is increasing. "
            "Thermal management recommended."
        )

    elif temp <= 45:

        return (
            "🟠 HIGH",
            "High battery temperature. "
            "Charging/power should be reduced."
        )

    else:

        return (
            "🔴 CRITICAL",
            "Critical battery temperature. "
            "Immediate thermal protection required."
        )


# ============================================================
# 10. SOC STATUS
# ============================================================

def soc_status(soc):

    if soc <= 5:

        return (
            "🔴 CRITICAL",
            "Extremely low SOC."
        )

    elif soc <= 15:

        return (
            "🟠 LOW",
            "Low battery. Charging recommended."
        )

    elif soc <= 20:

        return (
            "🟡 WARNING",
            "Battery SOC is approaching low level."
        )

    elif soc >= 95:

        return (
            "🟡 VERY HIGH",
            "Battery is nearly fully charged."
        )

    else:

        return (
            "🟢 NORMAL",
            "SOC is within normal operating range."
        )


# ============================================================
# 11. SOH STATUS
# ============================================================

def soh_status(soh):

    if soh < 60:

        return (
            "🔴 CRITICAL",
            "Battery degradation is severe."
        )

    elif soh < 75:

        return (
            "🟠 POOR",
            "Significant battery degradation detected."
        )

    elif soh < 85:

        return (
            "🟡 AGING",
            "Battery shows moderate degradation."
        )

    else:

        return (
            "🟢 GOOD",
            "Battery health is good."
        )


# ============================================================
# 12. TIRE SYSTEM
# ============================================================

def tire_system_status(
        fl,
        fr,
        rl,
        rr):

    pressures = [fl, fr, rl, rr]

    minimum = min(pressures)
    maximum = max(pressures)

    difference = maximum - minimum

    if minimum < 15:

        return (
            "🔴 CRITICAL",
            "One or more tires are critically under-inflated."
        )

    elif minimum < 25:

        return (
            "🟠 LOW PRESSURE",
            "One or more tires have low pressure."
        )

    elif difference > 4:

        return (
            "🟡 IMBALANCE",
            "Large pressure difference between tires."
        )

    elif maximum > 38:

        return (
            "🟡 HIGH PRESSURE",
            "One or more tires are over-inflated."
        )

    else:

        return (
            "🟢 NORMAL",
            "Tire pressures are within the modeled range."
        )


# ============================================================
# 13. MAIN EV PHYSICS MODEL
# ============================================================

def calculate_ev(
        soc,
        soh,
        battery_capacity,
        battery_temp,
        outdoor_temp,
        ac_temp,
        speed,
        payload,
        tire_fl,
        tire_fr,
        tire_rl,
        tire_rr,
        drive_mode):


    mode = DRIVE_MODES[drive_mode]


    # --------------------------------------------------------
    # TOTAL VEHICLE MASS
    # --------------------------------------------------------

    total_mass = (
        VEHICLE_MASS
        + payload
    )


    # --------------------------------------------------------
    # SPEED
    # --------------------------------------------------------

    speed_ms = (
        speed / 3.6
    )


    # --------------------------------------------------------
    # TIRE EFFECT
    # --------------------------------------------------------

    tire_factors = [

        tire_pressure_factor(tire_fl),

        tire_pressure_factor(tire_fr),

        tire_pressure_factor(tire_rl),

        tire_pressure_factor(tire_rr)
    ]

    average_tire_factor = np.mean(
        tire_factors
    )


    # --------------------------------------------------------
    # ROLLING RESISTANCE
    # --------------------------------------------------------

    base_crr = 0.010

    crr = (
        base_crr
        * average_tire_factor
    )

    F_rolling = (
        crr
        * total_mass
        * GRAVITY
    )


    # --------------------------------------------------------
    # AERODYNAMIC DRAG
    # --------------------------------------------------------

    F_aero = (

        0.5
        * AIR_DENSITY
        * CD
        * FRONTAL_AREA
        * speed_ms ** 2
    )


    # --------------------------------------------------------
    # TOTAL TRACTION FORCE
    # --------------------------------------------------------

    F_traction = (
        F_rolling
        + F_aero
    )


    # --------------------------------------------------------
    # MECHANICAL POWER
    # --------------------------------------------------------

    P_mechanical = (
        F_traction
        * speed_ms
    )


    # --------------------------------------------------------
    # ELECTRICAL TRACTION POWER
    # --------------------------------------------------------

    if speed > 0:

        P_traction = (

            P_mechanical
            /
            DRIVETRAIN_EFFICIENCY
        )

    else:

        P_traction = 0


    P_traction_kW = (
        P_traction / 1000
    )


    # --------------------------------------------------------
    # DRIVE MODE POWER EFFECT
    # --------------------------------------------------------

    P_traction_kW *= (
        mode["power_factor"]
    )


    # --------------------------------------------------------
    # HVAC
    # --------------------------------------------------------

    P_hvac = calculate_hvac_power(

        outdoor_temp,
        ac_temp,
        drive_mode
    )


    # --------------------------------------------------------
    # BATTERY THERMAL LOAD
    # --------------------------------------------------------

    P_thermal = battery_thermal_load(

        battery_temp,
        outdoor_temp
    )


    # --------------------------------------------------------
    # AUXILIARY LOAD
    # --------------------------------------------------------

    base_auxiliary = 0.40

    P_auxiliary = (
        base_auxiliary
        * mode["aux_factor"]
    )


    # --------------------------------------------------------
    # TOTAL BATTERY POWER
    # --------------------------------------------------------

    P_battery = (

        P_traction_kW
        + P_hvac
        + P_thermal
        + P_auxiliary
    )


    # --------------------------------------------------------
    # SOH EFFECT
    # --------------------------------------------------------

    soh_factor = max(
        0.50,
        min(1.0, soh / 100)
    )


    # --------------------------------------------------------
    # USABLE BATTERY CAPACITY
    # --------------------------------------------------------

    usable_capacity = (

        battery_capacity
        * soh_factor
    )


    # --------------------------------------------------------
    # REMAINING ENERGY
    # --------------------------------------------------------

    remaining_energy = (

        usable_capacity
        * soc
        / 100
    )


    # --------------------------------------------------------
    # ENERGY CONSUMPTION
    # --------------------------------------------------------

    if speed > 0:

        energy_per_km = (

            P_battery
            / speed
        )

    else:

        energy_per_km = 0


    # --------------------------------------------------------
    # RANGE
    # --------------------------------------------------------

    if energy_per_km > 0:

        total_range = (

            usable_capacity
            /
            energy_per_km
        )

        remaining_range = (

            remaining_energy
            /
            energy_per_km
        )

    else:

        total_range = 0
        remaining_range = 0


    # --------------------------------------------------------
    # BATTERY CURRENT
    # --------------------------------------------------------

    if P_battery > 0:

        battery_current = (

            P_battery * 1000
            /
            NOMINAL_PACK_VOLTAGE
        )

    else:

        battery_current = 0


    # ========================================================
    # BATTERY STRESS INDEX
    # ========================================================

    temperature_stress = min(

        100,

        abs(battery_temp - 25)
        * 3
    )


    speed_stress = min(

        100,

        max(0, speed - 90)
        * 0.7
    )


    payload_stress = (

        payload
        /
        750
        * 100
    )


    tire_stress = max(

        0,

        (average_tire_factor - 1)
        * 250
    )


    drive_stress = (

        0
        if drive_mode == "Eco"
        else
        10
        if drive_mode == "Normal"
        else
        20
    )


    battery_stress = (

        0.35 * temperature_stress

        + 0.20 * speed_stress

        + 0.15 * payload_stress

        + 0.15 * tire_stress

        + 0.15 * drive_stress
    )


    battery_stress = min(

        100,

        max(0, battery_stress)
    )


    # ========================================================
    # WARNING / ALERT SYSTEM
    # ========================================================

    alerts = []


    # Battery temperature

    temp_status, temp_message = (
        battery_temperature_status(
            battery_temp
        )
    )

    if battery_temp > 45:

        alerts.append(
            "🔴 CRITICAL: Battery temperature above 45°C."
        )

    elif battery_temp > 40:

        alerts.append(
            "🟠 WARNING: High battery temperature."
        )

    elif battery_temp < 0:

        alerts.append(
            "🔵 WARNING: Battery temperature below 0°C."
        )


    # SOC

    soc_level, soc_message = (
        soc_status(soc)
    )

    if soc <= 5:

        alerts.append(
            "🔴 CRITICAL: SOC extremely low."
        )

    elif soc <= 15:

        alerts.append(
            "🟠 WARNING: Low battery SOC."
        )


    # SOH

    soh_level, soh_message = (
        soh_status(soh)
    )

    if soh < 60:

        alerts.append(
            "🔴 CRITICAL: Battery SOH below 60%."
        )

    elif soh < 75:

        alerts.append(
            "🟠 WARNING: Significant battery degradation."
        )


    # Tire alerts

    tire_level, tire_message = (
        tire_system_status(

            tire_fl,
            tire_fr,
            tire_rl,
            tire_rr
        )
    )


    if tire_fl < 25:

        alerts.append(
            "🛞 FL: Low tire pressure."
        )

    if tire_fr < 25:

        alerts.append(
            "🛞 FR: Low tire pressure."
        )

    if tire_rl < 25:

        alerts.append(
            "🛞 RL: Low tire pressure."
        )

    if tire_rr < 25:

        alerts.append(
            "🛞 RR: Low tire pressure."
        )


    # Tire imbalance

    if max(
        tire_fl,
        tire_fr,
        tire_rl,
        tire_rr
    ) - min(
        tire_fl,
        tire_fr,
        tire_rl,
        tire_rr
    ) > 4:

        alerts.append(
            "🛞 WARNING: Tire pressure imbalance."
        )


    # Payload

    if payload > 600:

        alerts.append(
            "⚠️ WARNING: High vehicle payload."
        )


    # Speed

    if speed > 120:

        alerts.append(
            "⚠️ HIGH SPEED: Energy consumption significantly increased."
        )

    elif speed > 100:

        alerts.append(
            "🟡 Speed above efficient driving range."
        )


    # HVAC

    if abs(
        outdoor_temp - ac_temp
    ) > 15:

        alerts.append(
            "❄️ HVAC load is significantly increasing battery consumption."
        )


    # Battery stress

    if battery_stress >= 75:

        alerts.append(
            "🔴 CRITICAL: High combined battery stress."
        )

    elif battery_stress >= 50:

        alerts.append(
            "🟠 WARNING: Elevated battery stress."
        )


    # Sport mode

    if drive_mode == "Sport":

        alerts.append(
            "🏎️ SPORT MODE: Higher power demand and battery consumption."
        )


    # ========================================================
    # OVERALL STATUS
    # ========================================================

    if battery_temp > 45:

        overall = (
            "🔴 CRITICAL"
        )

    elif soc <= 5:

        overall = (
            "🔴 CRITICAL"
        )

    elif soh < 60:

        overall = (
            "🔴 CRITICAL"
        )

    elif battery_stress >= 75:

        overall = (
            "🔴 CRITICAL LOAD"
        )

    elif (
        battery_temp > 40
        or
        soc <= 15
        or
        battery_stress >= 50
    ):

        overall = (
            "🟠 WARNING"
        )

    elif (
        tire_level != "🟢 NORMAL"
    ):

        overall = (
            "🟡 CHECK TIRES"
        )

    else:

        overall = (
            "🟢 NORMAL"
        )


    # ========================================================
    # RECOMMENDED CHARGING CURRENT
    # ========================================================

    charging_current = 32.0


    if battery_temp > 35:

        charging_current *= 0.85


    if battery_temp > 40:

        charging_current *= 0.65


    if battery_temp > 45:

        charging_current *= 0.30


    if soh < 85:

        charging_current *= 0.90


    if soh < 75:

        charging_current *= 0.75


    if soc > 80:

        charging_current *= 0.75


    if soc > 90:

        charging_current *= 0.50


    if battery_stress > 70:

        charging_current *= 0.60


    # ========================================================
    # DRIVE RECOMMENDATION
    # ========================================================

    if battery_temp > 40:

        recommendation = (
            "Reduce load and speed. "
            "Allow battery cooling."
        )

    elif battery_stress > 70:

        recommendation = (
            "High battery stress. "
            "Use ECO mode."
        )

    elif speed > 110:

        recommendation = (
            "Reduce speed to approximately "
            "70–100 km/h for better efficiency."
        )

    elif drive_mode == "Sport":

        recommendation = (
            "Sport mode active. "
            "Expect increased energy consumption."
        )

    else:

        recommendation = (
            "Vehicle operating within normal conditions."
        )


    return {

        "traction_power": P_traction_kW,

        "hvac_power": P_hvac,

        "thermal_power": P_thermal,

        "aux_power": P_auxiliary,

        "battery_power": P_battery,

        "energy_per_km": energy_per_km,

        "battery_current": battery_current,

        "usable_capacity": usable_capacity,

        "remaining_energy": remaining_energy,

        "total_range": total_range,

        "remaining_range": remaining_range,

        "battery_stress": battery_stress,

        "charging_current": charging_current,

        "temp_status": temp_status,

        "temp_message": temp_message,

        "soc_status": soc_level,

        "soc_message": soc_message,

        "soh_status": soh_level,

        "soh_message": soh_message,

        "tire_status": tire_level,

        "tire_message": tire_message,

        "overall": overall,

        "alerts": alerts,

        "recommendation": recommendation
    }


# ============================================================
# 14. DASHBOARD UPDATE FUNCTION
# ============================================================

def update_dashboard(
        soc,
        soh,
        battery_capacity,
        battery_temp,
        outdoor_temp,
        ac_temp,
        speed,
        payload,
        tire_fl,
        tire_fr,
        tire_rl,
        tire_rr,
        drive_mode):


    result = calculate_ev(

        soc,
        soh,
        battery_capacity,
        battery_temp,
        outdoor_temp,
        ac_temp,
        speed,
        payload,
        tire_fl,
        tire_fr,
        tire_rl,
        tire_rr,
        drive_mode
    )


    # ========================================================
    # ALERT PANEL
    # ========================================================

    if len(result["alerts"]) == 0:

        alert_text = (
            "### 🟢 NO ACTIVE WARNINGS\n\n"
            "All monitored parameters are within "
            "the modeled operating limits."
        )

    else:

        alert_text = (
            "### ⚠️ ACTIVE WARNINGS & ALERTS\n\n"
        )

        for alert in result["alerts"]:

            alert_text += (
                f"- {alert}\n"
            )


    # ========================================================
    # MAIN STATUS
    # ========================================================

    status_text = f"""
# 🔋 SMART EV BMS STATUS

## {result["overall"]}

### Drive Mode
**{drive_mode}**

### Battery

| Parameter | Value |
|---|---:|
| SOC | {soc:.1f} % |
| SOH | {soh:.1f} % |
| Battery Temperature | {battery_temp:.1f} °C |
| Battery Stress | {result["battery_stress"]:.1f} % |
| Usable Capacity | {result["usable_capacity"]:.2f} kWh |
| Remaining Energy | {result["remaining_energy"]:.2f} kWh |

### Vehicle

| Parameter | Value |
|---|---:|
| Speed | {speed:.1f} km/h |
| Payload | {payload:.1f} kg |
| Outdoor Temperature | {outdoor_temp:.1f} °C |
| AC Temperature | {ac_temp:.1f} °C |

### Energy

| Parameter | Value |
|---|---:|
| Traction Power | {result["traction_power"]:.2f} kW |
| HVAC Power | {result["hvac_power"]:.2f} kW |
| Thermal Management | {result["thermal_power"]:.2f} kW |
| Auxiliary Power | {result["aux_power"]:.2f} kW |
| **Total Battery Power** | **{result["battery_power"]:.2f} kW** |
| Battery Current | {result["battery_current"]:.1f} A |
| Energy Consumption | {result["energy_per_km"]:.4f} kWh/km |

### Range

| Parameter | Value |
|---|---:|
| Estimated Total Range | **{result["total_range"]:.1f} km** |
| Estimated Remaining Range | **{result["remaining_range"]:.1f} km** |

### Charging

Recommended Charging Current:

**{result["charging_current"]:.1f} A**

### Driving Recommendation

**{result["recommendation"]}**
"""


    # ========================================================
    # TIRE PANEL
    # ========================================================

    tire_text = f"""
# 🛞 TIRE MONITORING

## Overall: {result["tire_status"]}

| Tire | Pressure | Status |
|---|---:|---|
| Front Left | {tire_fl:.1f} PSI | {individual_tire_status(tire_fl)} |
| Front Right | {tire_fr:.1f} PSI | {individual_tire_status(tire_fr)} |
| Rear Left | {tire_rl:.1f} PSI | {individual_tire_status(tire_rl)} |
| Rear Right | {tire_rr:.1f} PSI | {individual_tire_status(tire_rr)} |

**System:** {result["tire_message"]}
"""


    # ========================================================
    # BATTERY INDICATOR DATA
    # ========================================================

    indicator_data = pd.DataFrame({

        "Indicator": [

            "SOC",

            "SOH",

            "Battery Temperature",

            "Battery Stress",

            "Speed",

            "Payload",

            "Battery Power",

            "Remaining Range"
        ],

        "Value": [

            f"{soc:.1f} %",

            f"{soh:.1f} %",

            f"{battery_temp:.1f} °C",

            f"{result['battery_stress']:.1f} %",

            f"{speed:.1f} km/h",

            f"{payload:.1f} kg",

            f"{result['battery_power']:.2f} kW",

            f"{result['remaining_range']:.1f} km"
        ]
    })


    return (
        status_text,
        alert_text,
        tire_text,
        indicator_data
    )


# ============================================================
# 15. CREATE INTERACTIVE DASHBOARD
# ============================================================

with gr.Blocks(
    title="Smart EV BMS AI Dashboard"
) as dashboard:


    # ========================================================
    # HEADER
    # ========================================================

    gr.Markdown(
        """
# 🔋 SMART EV BMS
## AI-Based Battery & Vehicle Energy Management System

**Real-time interactive EV simulation**

Adjust the vehicle, battery, environmental and tire
parameters to observe their effect on battery power,
energy consumption, range and safety.
"""
    )


    # ========================================================
    # DRIVE MODE
    # ========================================================

    gr.Markdown(
        "## 🚗 Drive Mode"
    )

    drive_mode = gr.Radio(

        choices=[
            "Eco",
            "Normal",
            "Sport"
        ],

        value="Normal",

        label="Select Driving Mode"
    )


    # ========================================================
    # BATTERY
    # ========================================================

    gr.Markdown(
        "## 🔋 Battery Parameters"
    )

    with gr.Row():

        soc = gr.Slider(

            minimum=0,
            maximum=100,
            value=70,
            step=1,

            label="Battery SOC (%)"
        )

        soh = gr.Slider(

            minimum=50,
            maximum=100,
            value=95,
            step=1,

            label="Battery SOH (%)"
        )

        battery_capacity = gr.Slider(

            minimum=10,
            maximum=150,
            value=60,
            step=1,

            label="Battery Capacity (kWh)"
        )


    with gr.Row():

        battery_temp = gr.Slider(

            minimum=-20,
            maximum=60,
            value=30,
            step=1,

            label="Battery Temperature (°C)"
        )


    # ========================================================
    # ENVIRONMENT
    # ========================================================

    gr.Markdown(
        "## 🌡️ Environmental & HVAC Parameters"
    )

    with gr.Row():

        outdoor_temp = gr.Slider(

            minimum=-20,
            maximum=55,
            value=30,
            step=1,

            label="Outdoor Temperature (°C)"
        )

        ac_temp = gr.Slider(

            minimum=16,
            maximum=30,
            value=24,
            step=1,

            label="AC Set Temperature (°C)"
        )


    # ========================================================
    # VEHICLE
    # ========================================================

    gr.Markdown(
        "## 🚘 Vehicle Parameters"
    )

    with gr.Row():

        speed = gr.Slider(

            minimum=0,
            maximum=160,
            value=60,
            step=1,

            label="Vehicle Speed (km/h)"
        )

        payload = gr.Slider(

            minimum=0,
            maximum=750,
            value=100,
            step=5,

            label="Payload (kg)"
        )


    # ========================================================
    # TIRES
    # ========================================================

    gr.Markdown(
        "## 🛞 Individual Tire Pressure"
    )

    with gr.Row():

        tire_fl = gr.Slider(

            minimum=0,
            maximum=40,
            value=34,
            step=0.5,

            label="Front Left (PSI)"
        )

        tire_fr = gr.Slider(

            minimum=0,
            maximum=40,
            value=34,
            step=0.5,

            label="Front Right (PSI)"
        )


    with gr.Row():

        tire_rl = gr.Slider(

            minimum=0,
            maximum=40,
            value=34,
            step=0.5,

            label="Rear Left (PSI)"
        )

        tire_rr = gr.Slider(

            minimum=0,
            maximum=40,
            value=34,
            step=0.5,

            label="Rear Right (PSI)"
        )


    # ========================================================
    # ANALYZE BUTTON
    # ========================================================

    analyze = gr.Button(

        "⚡ ANALYZE EV",

        variant="primary"
    )


    # ========================================================
    # OUTPUT TABS
    # ========================================================

    with gr.Tabs():


        # ----------------------------------------------------
        # BMS STATUS
        # ----------------------------------------------------

        with gr.Tab(
            "🔋 BMS Status"
        ):

            status_output = gr.Markdown()


        # ----------------------------------------------------
        # ALERTS
        # ----------------------------------------------------

        with gr.Tab(
            "⚠️ Warnings & Alerts"
        ):

            alert_output = gr.Markdown()


        # ----------------------------------------------------
        # TIRES
        # ----------------------------------------------------

        with gr.Tab(
            "🛞 Tire Monitoring"
        ):

            tire_output = gr.Markdown()


        # ----------------------------------------------------
        # INDICATORS
        # ----------------------------------------------------

        with gr.Tab(
            "📊 Indicators"
        ):

            indicator_output = gr.Dataframe()


    # ========================================================
    # CALCULATE
    # ========================================================

    analyze.click(

        fn=update_dashboard,

        inputs=[

            soc,
            soh,
            battery_capacity,

            battery_temp,

            outdoor_temp,
            ac_temp,

            speed,
            payload,

            tire_fl,
            tire_fr,
            tire_rl,
            tire_rr,

            drive_mode
        ],

        outputs=[

            status_output,

            alert_output,

            tire_output,

            indicator_output
        ]
    )


# ============================================================
# 16. LAUNCH
# ============================================================

dashboard.launch(
    share=True,
    debug=True
)

In [ ]:
# ============================================================
# SMART EV BMS
# LIVE INTERACTIVE EV BMS + VEHICLE ENERGY MANAGEMENT SYSTEM
# GOOGLE COLAB VERSION
#
# FEATURES
# ------------------------------------------------------------
# • Live sliders
# • Automatic dependency calculations
# • Drive modes
# • Battery SOC / SOH
# • Battery temperature
# • Outdoor temperature
# • AC temperature
# • Vehicle speed
# • Payload
# • 4 individual tire pressure sensors
# • Sensor failure simulation
# • Sensor plausibility monitoring
# • Live warning / alert system
# • Live scrolling line chart
# • Range estimation
# • Battery power estimation
# • Battery stress index
# • Charging-current recommendation
# • Event/history recording
# ============================================================


# ============================================================
# 1. INSTALL
# ============================================================

!pip install -q gradio pandas numpy matplotlib


# ============================================================
# 2. IMPORTS
# ============================================================

import gradio as gr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datetime import datetime
import time


# ============================================================
# 3. VEHICLE PARAMETERS
# ============================================================

VEHICLE_MASS = 1600.0       # kg
CD = 0.29                   # aerodynamic drag coefficient
FRONTAL_AREA = 2.25         # m²
AIR_DENSITY = 1.225         # kg/m³
GRAVITY = 9.81              # m/s²

DRIVETRAIN_EFFICIENCY = 0.90

NOMINAL_PACK_VOLTAGE = 400.0

BASE_CRR = 0.010

REFERENCE_TIRE_PRESSURE = 34.0


# ============================================================
# 4. DRIVE MODE PARAMETERS
# ============================================================

DRIVE_MODES = {

    "Eco": {
        "power_factor": 0.88,
        "hvac_factor": 0.85,
        "aux_factor": 0.85,
        "stress_factor": 0.85
    },

    "Normal": {
        "power_factor": 1.00,
        "hvac_factor": 1.00,
        "aux_factor": 1.00,
        "stress_factor": 1.00
    },

    "Sport": {
        "power_factor": 1.15,
        "hvac_factor": 1.05,
        "aux_factor": 1.10,
        "stress_factor": 1.20
    }
}


# ============================================================
# 5. SENSOR LIMITS
# ============================================================

SENSOR_LIMITS = {

    "SOC": (0, 100),

    "SOH": (0, 100),

    "Battery Temperature": (-30, 70),

    "Outdoor Temperature": (-40, 60),

    "AC Temperature": (10, 35),

    "Speed": (0, 250),

    "Payload": (0, 1000),

    "Tire Pressure": (0, 60)
}


# ============================================================
# 6. LIVE HISTORY
# ============================================================

history = []

MAX_HISTORY = 150


# ============================================================
# 7. SENSOR FAILURE STATE
# ============================================================

FAILURE_MODES = [
    "None",
    "Stuck",
    "High Bias",
    "Low Bias",
    "Noise",
    "Intermittent",
    "Out of Range"
]


# ============================================================
# 8. SAFE VALUE FUNCTION
# ============================================================

def clamp(value, minimum, maximum):

    return max(
        minimum,
        min(maximum, value)
    )


# ============================================================
# 9. TIRE ROLLING RESISTANCE
# ============================================================

def tire_pressure_factor(pressure):

    pressure = clamp(
        pressure,
        0,
        40
    )

    if pressure < 15:

        return 1.60

    elif pressure < 20:

        return 1.40

    elif pressure < 25:

        return 1.22

    elif pressure < 30:

        return 1.10

    elif pressure < 32:

        return 1.04

    elif pressure <= 36:

        return 1.00

    elif pressure <= 38:

        return 1.03

    else:

        return 1.08


# ============================================================
# 10. SENSOR FAILURE SIMULATOR
# ============================================================

def apply_sensor_failure(
        value,
        failure_mode,
        sensor_name):

    if failure_mode == "None":

        return value


    if failure_mode == "Stuck":

        # Fixed previous-like value
        # deliberately deterministic
        return value


    if failure_mode == "High Bias":

        if sensor_name == "Tire Pressure":

            return value + 8

        elif sensor_name == "Temperature":

            return value + 10

        else:

            return value + 10


    if failure_mode == "Low Bias":

        if sensor_name == "Tire Pressure":

            return value - 8

        elif sensor_name == "Temperature":

            return value - 10

        else:

            return value - 10


    if failure_mode == "Noise":

        return value + np.random.normal(
            0,
            max(
                0.5,
                abs(value) * 0.05
            )
        )


    if failure_mode == "Intermittent":

        if np.random.random() < 0.20:

            return np.nan

        return value


    if failure_mode == "Out of Range":

        if sensor_name == "Tire Pressure":

            return 55

        elif sensor_name == "Temperature":

            return 80

        elif sensor_name == "SOC":

            return 120

        elif sensor_name == "SOH":

            return -10

        elif sensor_name == "Speed":

            return 300

        elif sensor_name == "Payload":

            return 1500

        else:

            return 100


    return value


# ============================================================
# 11. SENSOR DIAGNOSTICS
# ============================================================

def diagnose_sensor(
        value,
        sensor_type):

    if value is None or np.isnan(value):

        return (
            "🔴 FAILURE",
            "No valid sensor signal"
        )


    if sensor_type not in SENSOR_LIMITS:

        return (
            "🟢 OK",
            "Valid"
        )


    low, high = SENSOR_LIMITS[
        sensor_type
    ]


    if value < low or value > high:

        return (
            "🔴 FAILURE",
            "Out-of-range signal"
        )


    return (
        "🟢 OK",
        "Valid signal"
    )


# ============================================================
# 12. BATTERY TEMPERATURE STATUS
# ============================================================

def battery_temperature_status(temp):

    if temp < 0:

        return (
            "🔵 COLD",
            "Battery temperature is below 0°C."
        )

    elif temp < 10:

        return (
            "🔵 LOW",
            "Cold battery. Charging/power capability may be reduced."
        )

    elif temp < 20:

        return (
            "🟡 COOL",
            "Battery is below the preferred operating range."
        )

    elif temp <= 35:

        return (
            "🟢 NORMAL",
            "Battery temperature is in the preferred range."
        )

    elif temp <= 40:

        return (
            "🟡 WARM",
            "Battery temperature is increasing."
        )

    elif temp <= 45:

        return (
            "🟠 HIGH",
            "High battery temperature."
        )

    else:

        return (
            "🔴 CRITICAL",
            "Critical battery temperature."
        )


# ============================================================
# 13. SOC STATUS
# ============================================================

def soc_status(soc):

    if soc <= 5:

        return (
            "🔴 CRITICAL",
            "Extremely low SOC."
        )

    elif soc <= 15:

        return (
            "🟠 LOW",
            "Low battery SOC."
        )

    elif soc >= 95:

        return (
            "🟡 HIGH",
            "Battery nearly fully charged."
        )

    else:

        return (
            "🟢 NORMAL",
            "SOC is within normal operating limits."
        )


# ============================================================
# 14. SOH STATUS
# ============================================================

def soh_status(soh):

    if soh < 60:

        return (
            "🔴 CRITICAL",
            "Severe battery degradation."
        )

    elif soh < 75:

        return (
            "🟠 POOR",
            "Significant battery degradation."
        )

    elif soh < 85:

        return (
            "🟡 AGING",
            "Moderate battery degradation."
        )

    else:

        return (
            "🟢 GOOD",
            "Battery health is good."
        )


# ============================================================
# 15. HVAC MODEL
# ============================================================

def calculate_hvac_power(
        outdoor_temp,
        ac_temp,
        drive_mode):

    mode = DRIVE_MODES[
        drive_mode
    ]

    delta_t = abs(
        outdoor_temp - ac_temp
    )


    base_power = 0.60

    thermal_load = (
        0.08 * delta_t
    )


    extreme_load = 0


    if outdoor_temp >= 40:

        extreme_load = 1.0

    elif outdoor_temp >= 35:

        extreme_load = 0.5

    elif outdoor_temp <= 5:

        extreme_load = 0.8


    power = (

        base_power
        + thermal_load
        + extreme_load
    )


    power *= mode[
        "hvac_factor"
    ]


    return power


# ============================================================
# 16. BATTERY THERMAL LOAD
# ============================================================

def battery_thermal_load(
        battery_temp,
        outdoor_temp):

    if battery_temp < 0:

        load = 2.5

    elif battery_temp < 10:

        load = 1.5

    elif battery_temp < 20:

        load = 0.5

    elif battery_temp <= 35:

        load = 0

    elif battery_temp <= 40:

        load = 0.5

    elif battery_temp <= 45:

        load = 1.5

    else:

        load = 3.0


    if outdoor_temp > 40:

        load += 0.5


    return load


# ============================================================
# 17. MAIN EV MODEL
# ============================================================

def calculate_ev_model(

        soc,
        soh,
        battery_capacity,

        battery_temp,

        outdoor_temp,
        ac_temp,

        speed,
        payload,

        tire_fl,
        tire_fr,
        tire_rl,
        tire_rr,

        drive_mode):


    mode = DRIVE_MODES[
        drive_mode
    ]


    # --------------------------------------------------------
    # MASS
    # --------------------------------------------------------

    total_mass = (

        VEHICLE_MASS
        + payload
    )


    # --------------------------------------------------------
    # SPEED
    # --------------------------------------------------------

    speed_ms = (
        speed / 3.6
    )


    # --------------------------------------------------------
    # TIRE EFFECT
    # --------------------------------------------------------

    tire_factors = [

        tire_pressure_factor(
            tire_fl
        ),

        tire_pressure_factor(
            tire_fr
        ),

        tire_pressure_factor(
            tire_rl
        ),

        tire_pressure_factor(
            tire_rr
        )
    ]


    average_tire_factor = np.mean(
        tire_factors
    )


    # --------------------------------------------------------
    # ROLLING RESISTANCE
    # --------------------------------------------------------

    crr = (

        BASE_CRR
        * average_tire_factor
    )


    F_rolling = (

        crr
        * total_mass
        * GRAVITY
    )


    # --------------------------------------------------------
    # AERODYNAMIC DRAG
    # --------------------------------------------------------

    F_aero = (

        0.5
        * AIR_DENSITY
        * CD
        * FRONTAL_AREA
        * speed_ms ** 2
    )


    # --------------------------------------------------------
    # TRACTION POWER
    # --------------------------------------------------------

    total_force = (

        F_rolling
        + F_aero
    )


    mechanical_power = (

        total_force
        * speed_ms
    )


    if speed > 0:

        traction_power = (

            mechanical_power
            /
            DRIVETRAIN_EFFICIENCY
        )

    else:

        traction_power = 0


    traction_power_kW = (

        traction_power
        / 1000
    )


    traction_power_kW *= mode[
        "power_factor"
    ]


    # --------------------------------------------------------
    # HVAC
    # --------------------------------------------------------

    hvac_power = calculate_hvac_power(

        outdoor_temp,
        ac_temp,
        drive_mode
    )


    # --------------------------------------------------------
    # BATTERY THERMAL MANAGEMENT
    # --------------------------------------------------------

    thermal_power = battery_thermal_load(

        battery_temp,
        outdoor_temp
    )


    # --------------------------------------------------------
    # AUXILIARY LOAD
    # --------------------------------------------------------

    auxiliary_power = (

        0.40
        * mode["aux_factor"]
    )


    # --------------------------------------------------------
    # TOTAL BATTERY POWER
    # --------------------------------------------------------

    total_battery_power = (

        traction_power_kW
        + hvac_power
        + thermal_power
        + auxiliary_power
    )


    # --------------------------------------------------------
    # SOH EFFECT
    # --------------------------------------------------------

    soh_factor = clamp(
        soh / 100,
        0.50,
        1.00
    )


    usable_capacity = (

        battery_capacity
        * soh_factor
    )


    # --------------------------------------------------------
    # REMAINING ENERGY
    # --------------------------------------------------------

    remaining_energy = (

        usable_capacity
        * soc
        / 100
    )


    # --------------------------------------------------------
    # ENERGY CONSUMPTION
    # --------------------------------------------------------

    if speed > 1:

        energy_per_km = (

            total_battery_power
            /
            speed
        )

    else:

        # At standstill use a small placeholder
        # for HVAC/auxiliary consumption.
        energy_per_km = 0


    # --------------------------------------------------------
    # RANGE
    # --------------------------------------------------------

    if energy_per_km > 0:

        total_range = (

            usable_capacity
            /
            energy_per_km
        )

        remaining_range = (

            remaining_energy
            /
            energy_per_km
        )

    else:

        total_range = 0
        remaining_range = 0


    # --------------------------------------------------------
    # CURRENT
    # --------------------------------------------------------

    battery_current = (

        total_battery_power
        * 1000
        /
        NOMINAL_PACK_VOLTAGE
    )


    # --------------------------------------------------------
    # BATTERY STRESS
    # --------------------------------------------------------

    temperature_stress = min(

        100,

        abs(battery_temp - 25)
        * 3
    )


    speed_stress = min(

        100,

        max(0, speed - 90)
        * 0.7
    )


    payload_stress = (

        payload
        / 750
        * 100
    )


    tire_stress = min(

        100,

        max(
            0,
            (average_tire_factor - 1)
            * 250
        )
    )


    mode_stress = {

        "Eco": 0,

        "Normal": 10,

        "Sport": 20
    }[drive_mode]


    battery_stress = (

        0.35 * temperature_stress
        + 0.20 * speed_stress
        + 0.15 * payload_stress
        + 0.15 * tire_stress
        + 0.15 * mode_stress
    )


    battery_stress = clamp(

        battery_stress,
        0,
        100
    )


    return {

        "traction_power":
            traction_power_kW,

        "hvac_power":
            hvac_power,

        "thermal_power":
            thermal_power,

        "auxiliary_power":
            auxiliary_power,

        "battery_power":
            total_battery_power,

        "energy_per_km":
            energy_per_km,

        "battery_current":
            battery_current,

        "usable_capacity":
            usable_capacity,

        "remaining_energy":
            remaining_energy,

        "total_range":
            total_range,

        "remaining_range":
            remaining_range,

        "battery_stress":
            battery_stress
    }


# ============================================================
# 18. DEPENDENCY CALCULATION
# ============================================================
#
# This function creates calculated/derived values that respond
# automatically to the primary slider values.
#
# Examples:
#
# Higher payload
#      ↓
# Higher vehicle mass
#      ↓
# Higher rolling resistance
#      ↓
# Higher battery power
#      ↓
# Higher consumption
#      ↓
# Lower range
#
# Higher speed
#      ↓
# Higher aerodynamic drag
#      ↓
# Higher battery power
#      ↓
# Higher consumption
#
# ============================================================

def calculate_dependencies(

        soc,
        soh,
        battery_capacity,
        battery_temp,
        outdoor_temp,
        ac_temp,
        speed,
        payload,
        tire_fl,
        tire_fr,
        tire_rl,
        tire_rr,
        drive_mode):


    result = calculate_ev_model(

        soc,
        soh,
        battery_capacity,

        battery_temp,

        outdoor_temp,
        ac_temp,

        speed,
        payload,

        tire_fl,
        tire_fr,
        tire_rl,
        tire_rr,

        drive_mode
    )


    # --------------------------------------------------------
    # Estimated pack voltage
    # --------------------------------------------------------

    #
    # Approximate open-circuit relationship.
    #
    # This is NOT a cell-level electrochemical model.
    #

    estimated_voltage = (

        340
        + 60 * (soc / 100)
    )


    # Temperature correction

    if battery_temp > 40:

        estimated_voltage *= 0.98

    elif battery_temp < 10:

        estimated_voltage *= 0.97


    # --------------------------------------------------------
    # Estimated battery current
    # --------------------------------------------------------

    estimated_current = (

        result["battery_power"]
        * 1000
        /
        max(
            250,
            estimated_voltage
        )
    )


    # --------------------------------------------------------
    # Estimated battery power demand
    # --------------------------------------------------------

    power_demand = (

        result["battery_power"]
    )


    # --------------------------------------------------------
    # Automatic charging recommendation
    # --------------------------------------------------------

    charging_current = 32.0


    if battery_temp > 35:

        charging_current *= 0.85


    if battery_temp > 40:

        charging_current *= 0.65


    if battery_temp > 45:

        charging_current *= 0.30


    if soh < 85:

        charging_current *= 0.90


    if soh < 75:

        charging_current *= 0.75


    if soc > 80:

        charging_current *= 0.75


    if soc > 90:

        charging_current *= 0.50


    if result["battery_stress"] > 70:

        charging_current *= 0.60


    # --------------------------------------------------------
    # Recommended maximum speed
    # --------------------------------------------------------

    if battery_temp > 45:

        max_speed = 60

    elif battery_temp > 40:

        max_speed = 80

    elif result["battery_stress"] > 70:

        max_speed = 80

    else:

        max_speed = 120


    return {

        "result": result,

        "estimated_voltage":
            estimated_voltage,

        "estimated_current":
            estimated_current,

        "charging_current":
            charging_current,

        "max_speed":
            max_speed
    }


# ============================================================
# 19. WARNING ENGINE
# ============================================================

def generate_alerts(

        soc,
        soh,
        battery_temp,
        outdoor_temp,
        ac_temp,
        speed,
        payload,

        tire_fl,
        tire_fr,
        tire_rl,
        tire_rr,

        drive_mode,

        sensor_failures):


    alerts = []


    # --------------------------------------------------------
    # SENSOR FAILURE WARNINGS
    # --------------------------------------------------------

    for name, mode in sensor_failures.items():

        if mode != "None":

            alerts.append(

                f"🔴 SENSOR FAILURE: "
                f"{name} → {mode}"
            )


    # --------------------------------------------------------
    # BATTERY TEMPERATURE
    # --------------------------------------------------------

    if battery_temp > 45:

        alerts.append(
            "🔴 CRITICAL: Battery temperature > 45°C"
        )

    elif battery_temp > 40:

        alerts.append(
            "🟠 WARNING: Battery temperature > 40°C"
        )

    elif battery_temp < 0:

        alerts.append(
            "🔵 WARNING: Battery temperature < 0°C"
        )


    # --------------------------------------------------------
    # SOC
    # --------------------------------------------------------

    if soc <= 5:

        alerts.append(
            "🔴 CRITICAL: SOC ≤ 5%"
        )

    elif soc <= 15:

        alerts.append(
            "🟠 WARNING: SOC ≤ 15%"
        )


    # --------------------------------------------------------
    # SOH
    # --------------------------------------------------------

    if soh < 60:

        alerts.append(
            "🔴 CRITICAL: SOH < 60%"
        )

    elif soh < 75:

        alerts.append(
            "🟠 WARNING: SOH < 75%"
        )


    # --------------------------------------------------------
    # TIRES
    # --------------------------------------------------------

    tires = {

        "Front Left": tire_fl,

        "Front Right": tire_fr,

        "Rear Left": tire_rl,

        "Rear Right": tire_rr
    }


    for name, pressure in tires.items():

        if pressure < 15:

            alerts.append(
                f"🔴 {name}: CRITICAL tire pressure"
            )

        elif pressure < 25:

            alerts.append(
                f"🟠 {name}: LOW tire pressure"
            )

        elif pressure > 38:

            alerts.append(
                f"🟡 {name}: HIGH tire pressure"
            )


    if max(
        tires.values()
    ) - min(
        tires.values()
    ) > 4:

        alerts.append(
            "🟡 TPMS: Significant tire pressure imbalance"
        )


    # --------------------------------------------------------
    # PAYLOAD
    # --------------------------------------------------------

    if payload > 600:

        alerts.append(
            "🟠 WARNING: High payload"
        )


    # --------------------------------------------------------
    # SPEED
    # --------------------------------------------------------

    if speed > 120:

        alerts.append(
            "🟠 WARNING: High-speed energy consumption"
        )


    # --------------------------------------------------------
    # HVAC
    # --------------------------------------------------------

    if abs(
        outdoor_temp - ac_temp
    ) > 15:

        alerts.append(
            "🟡 HVAC: High climate-control energy demand"
        )


    # --------------------------------------------------------
    # DRIVE MODE
    # --------------------------------------------------------

    if drive_mode == "Sport":

        alerts.append(
            "🔵 INFO: Sport mode increases battery demand"
        )


    # --------------------------------------------------------
    # COMBINED THERMAL LOAD
    # --------------------------------------------------------

    if (

        battery_temp > 40
        and
        outdoor_temp > 35
    ):

        alerts.append(
            "🔴 COMBINED THERMAL LOAD: "
            "High battery + ambient temperature"
        )


    return alerts


# ============================================================
# 20. LIVE CHART
# ============================================================

def make_live_chart():

    if len(history) == 0:

        fig, ax = plt.subplots(
            figsize=(10, 5)
        )

        ax.set_title(
            "Live EV BMS History"
        )

        ax.set_xlabel(
            "Sample"
        )

        ax.set_ylabel(
            "Value"
        )

        ax.grid(
            True,
            alpha=0.3
        )

        return fig


    df = pd.DataFrame(
        history
    )


    fig, ax = plt.subplots(
        figsize=(10, 5)
    )


    ax.plot(

        df["sample"],

        df["soc"],

        label="SOC (%)"
    )


    ax.plot(

        df["sample"],

        df["battery_temp"],

        label="Battery Temp (°C)"
    )


    ax.plot(

        df["sample"],

        df["battery_stress"],

        label="Battery Stress (%)"
    )


    ax.plot(

        df["sample"],

        df["power"],

        label="Battery Power (kW)"
    )


    ax.set_title(
        "Live EV BMS Parameters"
    )

    ax.set_xlabel(
        "Live Sample"
    )

    ax.set_ylabel(
        "Value"
    )

    ax.legend()

    ax.grid(
        True,
        alpha=0.3
    )

    fig.tight_layout()


    return fig


# ============================================================
# 21. MAIN LIVE UPDATE
# ============================================================

def live_update(

        soc,
        soh,
        battery_capacity,

        battery_temp,

        outdoor_temp,
        ac_temp,

        speed,
        payload,

        tire_fl,
        tire_fr,
        tire_rl,
        tire_rr,

        drive_mode,

        fail_batt_temp,
        fail_soc,
        fail_soh,
        fail_speed,
        fail_payload,
        fail_outdoor,
        fail_ac,
        fail_fl,
        fail_fr,
        fail_rl,
        fail_rr):


    # ========================================================
    # SENSOR FAILURE MAP
    # ========================================================

    failures = {

        "Battery Temperature":
            fail_batt_temp,

        "SOC":
            fail_soc,

        "SOH":
            fail_soh,

        "Speed":
            fail_speed,

        "Payload":
            fail_payload,

        "Outdoor Temperature":
            fail_outdoor,

        "AC Temperature":
            fail_ac,

        "Front Left TPMS":
            fail_fl,

        "Front Right TPMS":
            fail_fr,

        "Rear Left TPMS":
            fail_rl,

        "Rear Right TPMS":
            fail_rr
    }


    # ========================================================
    # APPLY SENSOR FAILURES
    # ========================================================

    actual_battery_temp = apply_sensor_failure(

        battery_temp,
        fail_batt_temp,
        "Temperature"
    )


    actual_soc = apply_sensor_failure(

        soc,
        fail_soc,
        "SOC"
    )


    actual_soh = apply_sensor_failure(

        soh,
        fail_soh,
        "SOH"
    )


    actual_speed = apply_sensor_failure(

        speed,
        fail_speed,
        "Speed"
    )


    actual_payload = apply_sensor_failure(

        payload,
        fail_payload,
        "Payload"
    )


    actual_outdoor_temp = apply_sensor_failure(

        outdoor_temp,
        fail_outdoor,
        "Temperature"
    )


    actual_ac_temp = apply_sensor_failure(

        ac_temp,
        fail_ac,
        "Temperature"
    )


    actual_fl = apply_sensor_failure(

        tire_fl,
        fail_fl,
        "Tire Pressure"
    )


    actual_fr = apply_sensor_failure(

        tire_fr,
        fail_fr,
        "Tire Pressure"
    )


    actual_rl = apply_sensor_failure(

        tire_rl,
        fail_rl,
        "Tire Pressure"
    )


    actual_rr = apply_sensor_failure(

        tire_rr,
        fail_rr,
        "Tire Pressure"
    )


    # ========================================================
    # SENSOR VALIDATION
    # ========================================================

    sensor_values = {

        "Battery Temperature":
            actual_battery_temp,

        "SOC":
            actual_soc,

        "SOH":
            actual_soh,

        "Speed":
            actual_speed,

        "Payload":
            actual_payload,

        "Outdoor Temperature":
            actual_outdoor_temp,

        "AC Temperature":
            actual_ac_temp,

        "Front Left TPMS":
            actual_fl,

        "Front Right TPMS":
            actual_fr,

        "Rear Left TPMS":
            actual_rl,

        "Rear Right TPMS":
            actual_rr
    }


    sensor_rows = []


    for name, value in sensor_values.items():

        if "TPMS" in name:

            status, message = diagnose_sensor(
                value,
                "Tire Pressure"
            )

        elif name in [
            "Battery Temperature",
            "Outdoor Temperature",
            "AC Temperature"
        ]:

            status, message = diagnose_sensor(
                value,
                "Battery Temperature"
            )

        else:

            status, message = diagnose_sensor(
                value,
                name
            )


        sensor_rows.append({

            "Sensor":
                name,

            "Reading":
                "INVALID"
                if pd.isna(value)
                else round(
                    float(value),
                    2
                ),

            "Status":
                status,

            "Diagnostic":
                message
        })


    sensor_table = pd.DataFrame(
        sensor_rows
    )


    # ========================================================
    # HANDLE INVALID SENSOR VALUES
    # ========================================================

    # Use original slider value as fallback for physics
    # while reporting the sensor failure.

    if pd.isna(actual_battery_temp):

        actual_battery_temp = battery_temp

    if pd.isna(actual_soc):

        actual_soc = soc

    if pd.isna(actual_soh):

        actual_soh = soh

    if pd.isna(actual_speed):

        actual_speed = speed

    if pd.isna(actual_payload):

        actual_payload = payload

    if pd.isna(actual_outdoor_temp):

        actual_outdoor_temp = outdoor_temp

    if pd.isna(actual_ac_temp):

        actual_ac_temp = ac_temp

    if pd.isna(actual_fl):

        actual_fl = tire_fl

    if pd.isna(actual_fr):

        actual_fr = tire_fr

    if pd.isna(actual_rl):

        actual_rl = tire_rl

    if pd.isna(actual_rr):

        actual_rr = tire_rr


    # ========================================================
    # CALCULATE EV
    # ========================================================

    dependency = calculate_dependencies(

        actual_soc,
        actual_soh,
        battery_capacity,

        actual_battery_temp,

        actual_outdoor_temp,
        actual_ac_temp,

        actual_speed,
        actual_payload,

        actual_fl,
        actual_fr,
        actual_rl,
        actual_rr,

        drive_mode
    )


    result = dependency[
        "result"
    ]


    # ========================================================
    # ALERTS
    # ========================================================

    alerts = generate_alerts(

        actual_soc,
        actual_soh,

        actual_battery_temp,

        actual_outdoor_temp,
        actual_ac_temp,

        actual_speed,
        actual_payload,

        actual_fl,
        actual_fr,
        actual_rl,
        actual_rr,

        drive_mode,

        failures
    )


    # ========================================================
    # SENSOR FAILURE COUNT
    # ========================================================

    failure_count = sum(

        1
        for mode in failures.values()
        if mode != "None"
    )


    # ========================================================
    # RECORD HISTORY
    # ========================================================

    sample_number = (

        history[-1]["sample"] + 1
        if history
        else 1
    )


    history.append({

        "sample":
            sample_number,

        "time":
            datetime.now().strftime(
                "%H:%M:%S"
            ),

        "soc":
            actual_soc,

        "battery_temp":
            actual_battery_temp,

        "battery_stress":
            result["battery_stress"],

        "power":
            result["battery_power"],

        "range":
            result["remaining_range"],

        "speed":
            actual_speed
    })


    # Keep latest records only

    if len(history) > MAX_HISTORY:

        del history[
            :-MAX_HISTORY
        ]


    # ========================================================
    # OVERALL STATUS
    # ========================================================

    if failure_count > 0:

        overall_status = (
            "🔴 SENSOR FAULT"
        )

    elif actual_battery_temp > 45:

        overall_status = (
            "🔴 CRITICAL"
        )

    elif actual_soc <= 5:

        overall_status = (
            "🔴 CRITICAL"
        )

    elif result["battery_stress"] >= 75:

        overall_status = (
            "🔴 HIGH STRESS"
        )

    elif (

        actual_battery_temp > 40
        or
        actual_soc <= 15
        or
        result["battery_stress"] >= 50
    ):

        overall_status = (
            "🟠 WARNING"
        )

    else:

        overall_status = (
            "🟢 NORMAL"
        )


    # ========================================================
    # STATUS PANEL
    # ========================================================

    status = f"""

# 🔋 EV BMS — LIVE STATUS

## {overall_status}

### 🚗 Drive Mode: **{drive_mode}**

---

### Battery

| Parameter | Live Value |
|---|---:|
| SOC | **{actual_soc:.1f} %** |
| SOH | **{actual_soh:.1f} %** |
| Battery Temperature | **{actual_battery_temp:.1f} °C** |
| Battery Stress | **{result["battery_stress"]:.1f} %** |
| Battery Power | **{result["battery_power"]:.2f} kW** |
| Battery Current | **{dependency["estimated_current"]:.1f} A** |
| Estimated Pack Voltage | **{dependency["estimated_voltage"]:.1f} V** |

---

### Energy & Range

| Parameter | Value |
|---|---:|
| Energy Consumption | **{result["energy_per_km"]:.4f} kWh/km** |
| Usable Battery | **{result["usable_capacity"]:.2f} kWh** |
| Remaining Energy | **{result["remaining_energy"]:.2f} kWh** |
| Remaining Range | **{result["remaining_range"]:.1f} km** |
| Charging Current Limit | **{dependency["charging_current"]:.1f} A** |

---

### Vehicle

| Parameter | Value |
|---|---:|
| Speed | **{actual_speed:.1f} km/h** |
| Payload | **{actual_payload:.1f} kg** |
| Outdoor Temperature | **{actual_outdoor_temp:.1f} °C** |
| AC Temperature | **{actual_ac_temp:.1f} °C** |

---

### Live Sample

**#{sample_number}**

Sensor faults detected:

**{failure_count}**
"""


    # ========================================================
    # ALERT PANEL
    # ========================================================

    if not alerts:

        alert_text = """

# 🟢 SYSTEM HEALTHY

No active warnings or sensor faults detected.
"""

    else:

        alert_text = """

# ⚠️ LIVE WARNINGS & ALERTS

"""


        for alert in alerts:

            alert_text += (

                f"- {alert}\n"
            )


    # ========================================================
    # DEPENDENCY PANEL
    # ========================================================

    dependency_text = f"""

# 🔗 LIVE DEPENDENCY ANALYSIS

### Vehicle Mass

**{VEHICLE_MASS + actual_payload:.1f} kg**

Base vehicle:

{VEHICLE_MASS:.0f} kg

Payload:

{actual_payload:.1f} kg


### Tire Influence

Average pressure:

**{np.mean([
    actual_fl,
    actual_fr,
    actual_rl,
    actual_rr
]):.1f} PSI**


### Power Breakdown

- Traction: **{result["traction_power"]:.2f} kW**
- HVAC: **{result["hvac_power"]:.2f} kW**
- Thermal management: **{result["thermal_power"]:.2f} kW**
- Auxiliary: **{result["auxiliary_power"]:.2f} kW**

### Total

**{result["battery_power"]:.2f} kW**

---

### Automatic Dependencies

**Payload ↑**

→ Vehicle mass ↑

→ Rolling resistance ↑

→ Battery power ↑

→ Energy consumption ↑

→ Range ↓


**Speed ↑**

→ Aerodynamic drag ↑

→ Battery power ↑

→ Energy consumption ↑

→ Range ↓


**Battery temperature ↑**

→ Thermal load ↑

→ Battery stress ↑

→ Charging current limit ↓


**Outdoor temperature ↔ AC temperature difference ↑**

→ HVAC load ↑

→ Battery consumption ↑


**Tire pressure ↓**

→ Rolling resistance ↑

→ Energy consumption ↑


**SOH ↓**

→ Usable capacity ↓

→ Available range ↓
"""


    # ========================================================
    # LIVE CHART
    # ========================================================

    chart = make_live_chart()


    return (

        status,

        alert_text,

        sensor_table,

        dependency_text,

        chart
    )


# ============================================================
# 22. RESET HISTORY
# ============================================================

def reset_history():

    history.clear()

    return make_live_chart()


# ============================================================
# 23. BUILD GRADIO UI
# ============================================================

with gr.Blocks(

    title="Smart EV BMS Live Dashboard"
) as demo:


    # ========================================================
    # HEADER
    # ========================================================

    gr.Markdown("""

# 🔋 SMART EV BMS
## Live Interactive Battery & Vehicle Energy Management System

**Move any slider and the BMS recalculates automatically.**

The dashboard records the changing values and displays them
on the live trend chart.
""")


    # ========================================================
    # DRIVE MODE
    # ========================================================

    gr.Markdown(
        "## 🚗 Drive Mode"
    )


    drive_mode = gr.Radio(

        choices=[
            "Eco",
            "Normal",
            "Sport"
        ],

        value="Normal",

        label="Driving Mode"
    )


    # ========================================================
    # BATTERY
    # ========================================================

    gr.Markdown(
        "## 🔋 Battery"
    )


    with gr.Row():

        soc = gr.Slider(

            0,
            100,
            value=70,
            step=1,

            label="Battery SOC (%)"
        )


        soh = gr.Slider(

            50,
            100,
            value=95,
            step=1,

            label="Battery SOH (%)"
        )


        battery_capacity = gr.Slider(

            10,
            150,
            value=60,
            step=1,

            label="Battery Capacity (kWh)"
        )


    battery_temp = gr.Slider(

        -20,
        60,
        value=30,
        step=1,

        label="Battery Temperature (°C)"
    )


    # ========================================================
    # ENVIRONMENT
    # ========================================================

    gr.Markdown(
        "## 🌡️ Environment & HVAC"
    )


    with gr.Row():

        outdoor_temp = gr.Slider(

            -20,
            55,
            value=30,
            step=1,

            label="Outdoor Temperature (°C)"
        )


        ac_temp = gr.Slider(

            16,
            30,
            value=24,
            step=1,

            label="AC Set Temperature (°C)"
        )


    # ========================================================
    # VEHICLE
    # ========================================================

    gr.Markdown(
        "## 🚘 Vehicle"
    )


    with gr.Row():

        speed = gr.Slider(

            0,
            160,
            value=60,
            step=1,

            label="Speed (km/h)"
        )


        payload = gr.Slider(

            0,
            750,
            value=100,
            step=5,

            label="Payload (kg)"
        )


    # ========================================================
    # TIRES
    # ========================================================

    gr.Markdown(
        "## 🛞 TPMS — Individual Sensors"
    )


    with gr.Row():

        tire_fl = gr.Slider(

            0,
            40,
            value=34,
            step=0.5,

            label="Front Left — PSI"
        )


        tire_fr = gr.Slider(

            0,
            40,
            value=34,
            step=0.5,

            label="Front Right — PSI"
        )


    with gr.Row():

        tire_rl = gr.Slider(

            0,
            40,
            value=34,
            step=0.5,

            label="Rear Left — PSI"
        )


        tire_rr = gr.Slider(

            0,
            40,
            value=34,
            step=0.5,

            label="Rear Right — PSI"
        )


    # ========================================================
    # SENSOR FAILURE SIMULATION
    # ========================================================

    gr.Markdown(
        "## 🛠️ Sensor Failure Simulation"
    )


    gr.Markdown("""
Use these controls to test the BMS diagnostic system.

**None** = normal sensor

**Stuck** = sensor stuck at its current reading

**High/Low Bias** = systematic sensor error

**Noise** = unstable signal

**Intermittent** = occasional signal loss

**Out of Range** = impossible sensor reading
""")


    failure_choices = [

        "None",
        "Stuck",
        "High Bias",
        "Low Bias",
        "Noise",
        "Intermittent",
        "Out of Range"
    ]


    with gr.Row():

        fail_batt_temp = gr.Dropdown(

            failure_choices,
            value="None",

            label="Battery Temperature Sensor"
        )


        fail_soc = gr.Dropdown(

            failure_choices,
            value="None",

            label="SOC Sensor"
        )


        fail_soh = gr.Dropdown(

            failure_choices,
            value="None",

            label="SOH Sensor"
        )


    with gr.Row():

        fail_speed = gr.Dropdown(

            failure_choices,
            value="None",

            label="Speed Sensor"
        )


        fail_payload = gr.Dropdown(

            failure_choices,
            value="None",

            label="Payload Sensor"
        )


        fail_outdoor = gr.Dropdown(

            failure_choices,
            value="None",

            label="Outdoor Temperature Sensor"
        )


    with gr.Row():

        fail_ac = gr.Dropdown(

            failure_choices,
            value="None",

            label="AC Temperature Sensor"
        )


    with gr.Row():

        fail_fl = gr.Dropdown(

            failure_choices,
            value="None",

            label="FL TPMS"
        )


        fail_fr = gr.Dropdown(

            failure_choices,
            value="None",

            label="FR TPMS"
        )


        fail_rl = gr.Dropdown(

            failure_choices,
            value="None",

            label="RL TPMS"
        )


        fail_rr = gr.Dropdown(

            failure_choices,
            value="None",

            label="RR TPMS"
        )


    # ========================================================
    # OUTPUT TABS
    # ========================================================

    with gr.Tabs():


        with gr.Tab(
            "🔋 Live BMS"
        ):

            status_output = gr.Markdown()


        with gr.Tab(
            "⚠️ Alerts"
        ):

            alert_output = gr.Markdown()


        with gr.Tab(
            "🛠️ Sensor Diagnostics"
        ):

            sensor_output = gr.Dataframe()


        with gr.Tab(
            "🔗 Dependencies"
        ):

            dependency_output = gr.Markdown()


        with gr.Tab(
            "📈 Live Trend"
        ):

            chart_output = gr.Plot()


    # ========================================================
    # RESET
    # ========================================================

    reset_button = gr.Button(
        "♻️ Reset Live History"
    )


    # ========================================================
    # INPUT LIST
    # ========================================================

    all_inputs = [

        soc,
        soh,
        battery_capacity,

        battery_temp,

        outdoor_temp,
        ac_temp,

        speed,
        payload,

        tire_fl,
        tire_fr,
        tire_rl,
        tire_rr,

        drive_mode,

        fail_batt_temp,
        fail_soc,
        fail_soh,

        fail_speed,
        fail_payload,

        fail_outdoor,
        fail_ac,

        fail_fl,
        fail_fr,
        fail_rl,
        fail_rr
    ]


    # ========================================================
    # LIVE UPDATE
    # ========================================================
    #
    # .change() means the output updates whenever the user
    # changes a slider/dropdown instead of waiting for a
    # button.
    #

    for component in all_inputs:

        component.change(

            fn=live_update,

            inputs=all_inputs,

            outputs=[

                status_output,

                alert_output,

                sensor_output,

                dependency_output,

                chart_output
            ],

            queue=True
        )


    # ========================================================
    # RESET HISTORY
    # ========================================================

    reset_button.click(

        fn=reset_history,

        outputs=chart_output
    )


# ============================================================
# 24. INITIAL CALCULATION
# ============================================================

initial_output = live_update(

    70,
    95,
    60,

    30,

    30,
    24,

    60,
    100,

    34,
    34,
    34,
    34,

    "Normal",

    "None",
    "None",
    "None",

    "None",
    "None",

    "None",
    "None",

    "None",
    "None",
    "None",
    "None"
)


# ============================================================
# 25. LAUNCH
# ============================================================

demo.launch(

    share=True,

    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b8dad3deda29939978.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
